# Customer Churn Prediction Using Machine Learning

This notebook predicts whether a telecom customer is likely to leave the company.

**Workflow:** load data → inspect data → clean and encode features → explore relationships → split data → scale features → apply PCA → train models → evaluate models → predict for a new customer.

**Important:** Keep `WA_Fn-UseC_-Telco-Customer-Churn.csv` in the same folder as this notebook.

## 1. Installations and imports

Run the installation command only if an import fails. In VS Code, make sure the selected Jupyter kernel is the Python environment where these packages are installed.

In [ ]:
# If required, uncomment and run this line:
# %pip install numpy pandas matplotlib seaborn scikit-learn xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost is not installed. The other models will still run.")

print("Imports completed successfully.")

## 2. Load the dataset

`Path.cwd()` shows the folder from which the notebook is running. The CSV is expected in that folder. If it is elsewhere, update `DATA_PATH` to its full path.

`pd.read_csv()` loads the CSV into a pandas DataFrame (a table of rows and columns).

In [ ]:
DATA_PATH = Path.cwd() / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Current working folder:", Path.cwd())
print("Dataset path:", DATA_PATH)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "CSV not found. Put the CSV in the same folder as the notebook "
        "or update DATA_PATH to the correct file location."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Understand the dataset

- `head()` previews the first rows.
- `shape` gives (number of rows, number of columns).
- `info()` shows column names, data types and non-null counts.
- `describe()` summarizes numerical columns.
- `value_counts()` shows how many customers belong to each target class.

In [ ]:
df.info()
display(df.describe(include="all").T)
print("\nChurn counts:")
print(df["Churn"].value_counts())
print("\nChurn proportions:")
print(df["Churn"].value_counts(normalize=True).round(3))

## 4. Data cleaning and preprocessing

Machine-learning algorithms require numeric inputs. This dataset contains text categories such as `Contract` and `InternetService`.

Steps:
1. Remove `customerID`: it is an identifier, not a useful behavioural predictor.
2. Convert `TotalCharges` to numeric. Blank strings become missing values (`NaN`).
3. Fill missing `TotalCharges` values with the column mean.
4. Convert target `Churn`: Yes → 1, No → 0.
5. Convert categorical input columns into numeric indicator columns using one-hot encoding.

`drop_first=True` omits one category from each categorical variable to reduce redundant columns.

In [ ]:
data = df.copy()

# Remove the unique customer identifier
data = data.drop(columns=["customerID"])

# Convert TotalCharges from text to numeric; blank entries become NaN
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
missing_total = data["TotalCharges"].isna().sum()
print("Missing TotalCharges before filling:", missing_total)

# Fill missing values with the mean
data["TotalCharges"] = data["TotalCharges"].fillna(data["TotalCharges"].mean())

# Convert the target to binary values
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

# One-hot encode categorical input features
categorical_columns = data.drop(columns=["Churn"]).select_dtypes(include=["object"]).columns
cleaned_dataframe = pd.get_dummies(
    data, columns=categorical_columns, drop_first=True, dtype=int
)

print("Cleaned data shape:", cleaned_dataframe.shape)
print("Remaining missing values:", cleaned_dataframe.isna().sum().sum())
display(cleaned_dataframe.head())

## 5. Exploratory Data Analysis (EDA): correlation

Correlation measures the direction and strength of a linear relationship between numeric variables. Values range from -1 to +1:
- close to +1: positive association
- close to -1: negative association
- close to 0: weak linear association

This heatmap focuses on `tenure`, `MonthlyCharges`, `TotalCharges` and `Churn`. Correlation does not prove causation.

In [ ]:
target_attributes = ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
correlation_matrix = cleaned_dataframe[target_attributes].corr()

print(correlation_matrix["Churn"].sort_values(ascending=False))

plt.figure(figsize=(7, 5))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f",
            linewidths=0.5, vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Separate features and target; split the data

- `X` contains the customer information used to make predictions.
- `y` contains the answer the model must learn (`Churn`).
- 80% of records are used for training and 20% for testing.
- `stratify=y` keeps approximately the same Yes/No proportion in both sets.
- `random_state=42` makes the split reproducible.

The test set is held back to evaluate how well the model works on unseen records.

In [ ]:
X = cleaned_dataframe.drop(columns=["Churn"])
y = cleaned_dataframe["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("\nTraining churn proportion:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTesting churn proportion:")
print(y_test.value_counts(normalize=True).round(3))

## 7. Feature scaling and PCA

**StandardScaler** transforms each feature so it has approximately mean 0 and standard deviation 1. It is useful when features have different units or ranges.

Important: fit the scaler only on the training data. Then use that fitted scaler to transform the test data. This avoids leaking information from the test set into training.

**PCA (Principal Component Analysis)** creates a smaller set of new numeric features called principal components. Here we keep 5 components. PCA is used for the Logistic Regression model in this notebook; Random Forest and XGBoost use the scaled original features.

In [ ]:
scaler_transformer = StandardScaler()
X_train_scaled = scaler_transformer.fit_transform(X_train)
X_test_scaled = scaler_transformer.transform(X_test)

pca_compressor = PCA(n_components=5, random_state=RANDOM_STATE)
X_train_pca = pca_compressor.fit_transform(X_train_scaled)
X_test_pca = pca_compressor.transform(X_test_scaled)

variance_captured = pca_compressor.explained_variance_ratio_.sum() * 100
print(f"Variance retained by 5 principal components: {variance_captured:.2f}%")

## 8. Train and evaluate multiple classification models

Three algorithms are considered:

1. **Logistic Regression:** a linear classification algorithm. It estimates the probability of churn and is a useful baseline. In this notebook it receives the 5 PCA components.
2. **Random Forest:** combines many decision trees and aggregates their predictions. It can learn non-linear relationships. It receives the scaled original features.
3. **XGBoost:** a gradient-boosted tree algorithm that builds trees sequentially to correct earlier errors. It is optional; the notebook continues if the package is not installed.

Metrics:
- **Accuracy:** fraction of all predictions that are correct.
- **Precision:** among customers predicted to churn, fraction who actually churned.
- **Recall:** among actual churners, fraction correctly identified.
- **F1-score:** harmonic mean of precision and recall.

For churn work, recall is important because a false negative means a customer who leaves was not flagged. Precision also matters because retention outreach has a cost.

In [ ]:
def evaluate_model(model, model_name, X_tr, X_te, y_tr, y_te):
    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)

    results = {
        "Model": model_name,
        "Train Accuracy": accuracy_score(y_tr, train_pred),
        "Test Accuracy": accuracy_score(y_te, test_pred),
        "Precision": precision_score(y_te, test_pred, zero_division=0),
        "Recall": recall_score(y_te, test_pred, zero_division=0),
        "F1-score": f1_score(y_te, test_pred, zero_division=0)
    }

    print("\n" + "=" * 60)
    print(model_name)
    print("=" * 60)
    print(pd.Series(results).to_string())
    print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y_te, test_pred))
    print("\nClassification report:")
    print(classification_report(y_te, test_pred, zero_division=0))
    return results

models = {}
results_list = []

# Logistic Regression uses PCA-transformed data
logistic_agent = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logistic_agent.fit(X_train_pca, y_train)
models["Logistic Regression"] = {
    "model": logistic_agent, "input_type": "pca"
}
results_list.append(evaluate_model(
    logistic_agent, "Logistic Regression (PCA)",
    X_train_pca, X_test_pca, y_train, y_test
))

# Random Forest uses scaled original features
random_forest_agent = RandomForestClassifier(
    n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1,
    class_weight="balanced"
)
random_forest_agent.fit(X_train_scaled, y_train)
models["Random Forest"] = {
    "model": random_forest_agent, "input_type": "scaled"
}
results_list.append(evaluate_model(
    random_forest_agent, "Random Forest",
    X_train_scaled, X_test_scaled, y_train, y_test
))

# XGBoost is optional
if HAS_XGB:
    xgboost_agent = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        random_state=RANDOM_STATE, eval_metric="logloss",
        n_jobs=-1
    )
    xgboost_agent.fit(X_train_scaled, y_train)
    models["XGBoost"] = {"model": xgboost_agent, "input_type": "scaled"}
    results_list.append(evaluate_model(
        xgboost_agent, "XGBoost",
        X_train_scaled, X_test_scaled, y_train, y_test
    ))
else:
    print("Skipping XGBoost because it is not installed.")

results_df = pd.DataFrame(results_list).sort_values("F1-score", ascending=False)
display(results_df)

## 9. Compare models and select one for the demonstration

The table sorts models by test F1-score. This is a simple demonstration selection rule, not proof that one model is best for every business. In a real retention system, the decision threshold should be chosen based on the cost of missed churners and unnecessary retention offers.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
# Remove the PCA suffix from the displayed Logistic Regression label
best_key = "Logistic Regression" if best_model_name.startswith("Logistic Regression") else best_model_name
best_model = models[best_key]["model"]
best_input_type = models[best_key]["input_type"]

print("Selected model:", best_key)
print("Input transformation:", best_input_type)

## 10. Predict churn for a new customer

The new customer's input must use the same original feature names and category spellings as the training dataset.

The function below:
1. creates a one-row DataFrame,
2. one-hot encodes its categorical columns,
3. aligns its columns to the training feature columns,
4. applies the already-fitted scaler,
5. applies PCA only if the selected model was trained on PCA features,
6. returns the predicted class and estimated churn probability.

This alignment step is essential because a new customer may not have every category represented in their one-row record.

In [ ]:
def predict_new_customer_churn(customer_data):
    new_df = pd.DataFrame([customer_data])

    # Match the categorical encoding used during training
    new_df_encoded = pd.get_dummies(new_df, dtype=int)
    new_df_aligned = new_df_encoded.reindex(columns=X.columns, fill_value=0)

    # Apply the already-fitted transformations
    new_scaled = scaler_transformer.transform(new_df_aligned)

    if best_input_type == "pca":
        model_input = pca_compressor.transform(new_scaled)
    else:
        model_input = new_scaled

    prediction = int(best_model.predict(model_input)[0])
    probability = float(best_model.predict_proba(model_input)[0][1] * 100)

    print("\nCUSTOMER CHURN PREDICTION")
    print("-" * 40)
    print(f"Estimated churn probability: {probability:.2f}%")
    print("Prediction:", "Likely to churn (Yes)" if prediction == 1 else "Not predicted to churn (No)")
    return prediction, probability

### 10A. Test using an example customer

The values below are an illustrative example, not a real customer's record. Change the values to experiment with different profiles.

In [ ]:
sample_customer = {
    "gender": "Female",
    "SeniorCitizen": 1,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 105.50,
    "TotalCharges": 211.00
}

sample_prediction, sample_probability = predict_new_customer_churn(sample_customer)

## 11. Interactive customer tester (optional)

Run this cell if you want to enter customer details in the notebook output. It asks for tenure, monthly charges and total charges; the other fields use a fixed example profile. For a more realistic prediction, edit all fields in `sample_customer` above.

In [ ]:
def launch_interactive_tester():
    try:
        user_tenure = int(input("Enter customer tenure in months (0-72): "))
        user_monthly = float(input("Enter monthly charges: "))
        user_total = float(input("Enter total charges: "))

        if not 0 <= user_tenure <= 72:
            raise ValueError("Tenure must be between 0 and 72 months.")
        if user_monthly < 0 or user_total < 0:
            raise ValueError("Charges cannot be negative.")

        customer = {
            "gender": "Male", "SeniorCitizen": 0, "Partner": "No",
            "Dependents": "No", "tenure": user_tenure,
            "PhoneService": "Yes", "MultipleLines": "No",
            "InternetService": "DSL", "OnlineSecurity": "No",
            "OnlineBackup": "No", "DeviceProtection": "No",
            "TechSupport": "No", "StreamingTV": "No",
            "StreamingMovies": "No", "Contract": "Month-to-month",
            "PaperlessBilling": "No", "PaymentMethod": "Mailed check",
            "MonthlyCharges": user_monthly, "TotalCharges": user_total
        }
        return predict_new_customer_churn(customer)
    except ValueError as error:
        print("Please enter valid values:", error)

# Uncomment to launch the input prompts:
# launch_interactive_tester()

## 12. Final conclusion

This project demonstrates a complete binary classification workflow for telecom customer churn. It loads and explores customer records, cleans missing and categorical data, transforms the features, trains classification models and evaluates them using multiple metrics. The selected model can then estimate churn probability for a new customer profile.

**Interpretation caution:** a prediction is an estimate, not a guarantee that a customer will leave. Model quality should be judged using test metrics and business costs, especially false negatives and false positives.